# Compute features and labels (Impect)

Requires `spadl-impect.h5` from notebook 1 or `private/impect-pipeline/build_spadl_h5.py`.

Paths come from your iteration config (private config or `docs/.../impect_open_data.example.json`).

In [ ]:
%load_ext autoreload
%autoreload 2
import json
import warnings
from pathlib import Path
import pandas as pd
import tqdm
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

PROJECT_ROOT = Path('..').resolve()
CONFIG_CANDIDATES = [
    PROJECT_ROOT / 'private/impect-pipeline/config/my_iteration.json',
    PROJECT_ROOT / 'docs/documentation/data/impect_open_data.example.json',
]
CONFIG_PATH = next((p for p in CONFIG_CANDIDATES if p.is_file()), CONFIG_CANDIDATES[-1])
with open(CONFIG_PATH) as f:
    CFG = json.load(f)
OUT = PROJECT_ROOT / 'data/impect' / CFG['output_basename']
spadl_h5 = OUT / 'spadl-impect.h5'
features_h5 = OUT / 'features.h5'
labels_h5 = OUT / 'labels.h5'
predictions_h5 = OUT / 'predictions.h5'
print('config:', CONFIG_PATH)
print('output:', OUT)

def games_with_actions(path):
    with pd.HDFStore(path) as store:
        games = store['games']
        ids = {int(k.rsplit('_', 1)[-1]) for k in store.keys() if k.startswith('/actions/game_')}
    return games[games.game_id.isin(ids)].sort_values('game_date').reset_index(drop=True)

import socceraction.spadl as spadl
import socceraction.vaep.features as fs
import socceraction.vaep.labels as lab

In [ ]:
games = games_with_actions(spadl_h5)
print(len(games), "games with SPADL")

## Features

In [ ]:
xfns = [
    fs.actiontype, fs.actiontype_onehot, fs.bodypart_onehot,
    fs.result, fs.result_onehot, fs.goalscore,
    fs.startlocation, fs.endlocation, fs.movement, fs.space_delta,
    fs.startpolar, fs.endpolar, fs.team, fs.time_delta,
]
with pd.HDFStore(spadl_h5) as spadlstore, pd.HDFStore(features_h5, 'w') as featurestore:
    for game in tqdm.tqdm(list(games.itertuples()), desc='features'):
        actions = spadlstore[f'actions/game_{game.game_id}']
        gs = fs.gamestates(spadl.add_names(actions), 3)
        gs = fs.play_left_to_right(gs, game.home_team_id)
        X = pd.concat([fn(gs) for fn in xfns], axis=1)
        featurestore.put(f'game_{game.game_id}', X, format='table')


## Labels

In [ ]:
yfns = [lab.scores, lab.concedes, lab.goal_from_shot]
with pd.HDFStore(spadl_h5) as spadlstore, pd.HDFStore(labels_h5, 'w') as labelstore:
    for game in tqdm.tqdm(list(games.itertuples()), desc='labels'):
        actions = spadlstore[f'actions/game_{game.game_id}']
        Y = pd.concat([fn(spadl.add_names(actions)) for fn in yfns], axis=1)
        labelstore.put(f'game_{game.game_id}', Y, format='table')
